# ML-08 - Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Bikram-Mondal3/flyrank-ML/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

Lane: **Content Refresh / refresh prioritization**.

This notebook trains an honest first model for the Content Refresh classification task. It preserves the Week 4 transparent baseline rule, evaluates the baseline and model on the same client-held-out rows, and checks for target leakage before training.


## 1. Method Choice And Why

I will use a Random Forest Classifier for the Content Refresh lane. The model predicts whether a content item is in the declining-content proxy class, which is used as a decision-support signal for refresh prioritization.

Random Forest is appropriate because the inputs mix numeric performance, keyword, content, and freshness signals, and the useful pattern may not be purely linear. It is useful beyond the Week 4 hand-written baseline because it can learn interactions among those signals rather than applying one fixed score formula. Complexity alone is not the goal: the model is kept modest, compared against the Week 4 rule on the same rows, and interpreted through permutation importance and error analysis.

This model does **not** predict Google's ranking algorithm and does not prove that refreshing a page will improve performance. It only learns associations in the available Content Refresh data.


In [1]:
# Imports and data loading.
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

RANDOM_STATE = 42

candidate_paths = [
    Path("data/raw/content_refresh_anonymized.csv"),
    Path("../../data/raw/content_refresh_anonymized.csv"),
    Path("content_refresh_anonymized.csv"),
]
DATA_PATH = next((p for p in candidate_paths if p.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError("Could not find content_refresh_anonymized.csv from this notebook location.")

df = pd.read_csv(DATA_PATH)

print(f"Data path: {DATA_PATH}")
print(f"Rows loaded: {len(df):,}")
print(f"Columns loaded: {len(df.columns):,}")


Data path: ..\..\data\raw\content_refresh_anonymized.csv
Rows loaded: 30,000
Columns loaded: 44


## 2. Target, Features, And Leakage Audit

The target/proxy follows the project data dictionary: `is_declining_label = 1` when `trend_direction == "down"`, otherwise `0`. The label source is the trend calculation, so `trend_direction`, `trend_pct`, and the 30-day trend-window columns are excluded from the model features.

The feature list uses only decision-moment content metadata, keyword context, current 90-day aggregate activity, current rates, position, and freshness fields. IDs are kept only for grouping and joins, never as model inputs. The model does not use future-window columns, the target column, target-derived columns, or the fields directly used to construct the target.


In [2]:
# Target/proxy from the established project data dictionary.
df = df.copy()
df["is_declining_label"] = df["trend_direction"].astype(str).str.lower().eq("down").astype(int)
target_col = "is_declining_label"

numeric_features = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "impressions_90d",
    "clicks_90d",
    "pageviews_90d",
    "sessions_90d",
    "users_90d",
    "engaged_sessions_90d",
    "ai_sessions_90d",
    "scroll_events_90d",
    "days_with_impressions",
    "days_with_sessions",
    "content_age_days",
    "age_tier_order",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct",
]

categorical_features = [
    "competition_level",
    "content_type",
    "main_intent",
    "age_tier",
    "freshness_tier",
    "word_count_tier",
    "char_count_tier",
    "impression_tier",
    "position_tier",
]

feature_cols = [col for col in numeric_features + categorical_features if col in df.columns]
X = df[feature_cols].copy()
y = df[target_col].copy()

target_source_columns = {
    "trend_direction",
    "trend_pct",
    "impressions_last_30d",
    "impressions_prev_30d",
}
future_or_outcome_columns = {
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d",
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d",
    "declined_second_half",
    "refresh_priority",
}
label_derived_columns = {
    "is_declining_label",
    "trend_direction",
    "trend_pct",
    "label_derived_decline_signal",
    "refresh_priority",
}
identifier_columns = {"content_id", "client_id", "client_hash_id", "content_hash_id"}

leakage_report = {
    "target_in_features": [col for col in feature_cols if col == target_col],
    "target_source_features": sorted(target_source_columns.intersection(feature_cols)),
    "future_or_outcome_features": sorted(future_or_outcome_columns.intersection(feature_cols)),
    "label_derived_features": sorted(label_derived_columns.intersection(feature_cols)),
    "identifier_features": sorted(identifier_columns.intersection(feature_cols)),
}

print("Target definition: is_declining_label = 1 when trend_direction == 'down', else 0")
print("Positive class rate:", round(y.mean(), 4))
print("Feature count:", len(feature_cols))
print("Feature list:")
print(feature_cols)
print("\nLeakage audit:")
for check_name, columns in leakage_report.items():
    print(f"{check_name}: {columns}")

assert y.nunique() == 2, "Target must contain both classes."
assert all(len(columns) == 0 for columns in leakage_report.values()), "Leakage audit failed."
print("\nLeakage check passed: no target, target-source, future-window, label-derived, or ID columns are in X.")


Target definition: is_declining_label = 1 when trend_direction == 'down', else 0
Positive class rate: 0.5421
Feature count: 32
Feature list:
['search_volume', 'competition', 'cpc', 'word_count', 'char_count', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'content_age_days', 'age_tier_order', 'days_since_last_update', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'competition_level', 'content_type', 'main_intent', 'age_tier', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'impression_tier', 'position_tier']

Leakage audit:
target_in_features: []
target_source_features: []
future_or_outcome_features: []
label_derived_features: []
identifier_features: []

Leakage check passed: no target, target-source, future-window, label-derived, or ID columns are in X.


## 3. Split Design

I use a grouped train/test split by `client_id`, holding out 25% of clients for testing. Pages from the same client can share topic mix, publishing practices, and measurement patterns, so a client-level holdout is more honest than randomly splitting pages from the same client into both train and test.

The Week 4 baseline and Week 5 model are evaluated on the exact same held-out rows against the same target and same metrics.


In [3]:
groups = df["client_id"]
splitter = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=RANDOM_STATE)
train_idx, test_idx = next(splitter.split(X, y, groups=groups))

X_train = X.iloc[train_idx].copy()
X_test = X.iloc[test_idx].copy()
y_train = y.iloc[train_idx].copy()
y_test = y.iloc[test_idx].copy()
train_df = df.iloc[train_idx].copy()
test_df = df.iloc[test_idx].copy()

train_clients = set(train_df["client_id"])
test_clients = set(test_df["client_id"])
client_overlap = train_clients.intersection(test_clients)

split_summary = pd.DataFrame(
    [
        {"split": "train", "rows": len(train_df), "clients": len(train_clients), "positive_rate": y_train.mean(), "negative_rate": 1 - y_train.mean()},
        {"split": "test", "rows": len(test_df), "clients": len(test_clients), "positive_rate": y_test.mean(), "negative_rate": 1 - y_test.mean()},
    ]
)

print("Split rationale: grouped holdout by client_id to avoid same-client page leakage.")
print(f"Train size: {len(train_df):,}")
print(f"Test size: {len(test_df):,}")
print(f"Client overlap between train and test: {len(client_overlap)}")
print("\nClass distribution by split:")
display(split_summary)

assert len(client_overlap) == 0, "Grouped split failed: at least one client is in both train and test."
assert y_train.nunique() == 2 and y_test.nunique() == 2, "Both splits must contain both target classes."


Split rationale: grouped holdout by client_id to avoid same-client page leakage.
Train size: 22,885
Test size: 7,115
Client overlap between train and test: 0

Class distribution by split:


,split,rows,clients,positive_rate,negative_rate
0,train,22885,24,0.550011,0.449989
1,test,7115,8,0.516514,0.483486


## 4. Train And Compare Against Week 4 Baseline

The baseline is the actual Week 4 transparent rule: staleness, current visibility, high keyword demand, low CTR relative to position bucket, and a small visibility component are combined into `baseline_score`. To convert that ranking score into a classification prediction for this notebook, I set the review threshold at the 75th percentile of Week 4 scores on the training rows, then apply that fixed threshold to the held-out test rows.

This keeps the Week 4 scoring logic intact while avoiding a threshold chosen from the test labels or test distribution. Baseline and model are compared on the same test rows using precision, recall, F1, and ROC-AUC.


In [4]:
def add_week4_baseline_scores(all_rows, fit_rows):
    scored = all_rows.copy()
    fit = fit_rows.copy()

    demand_threshold = fit["search_volume"].quantile(0.75)
    visibility_threshold = 300
    position_ctr_threshold = (
        fit[fit["impressions_90d"] >= 100]
        .groupby("position_tier", observed=False)["ctr"]
        .median()
        .to_dict()
    )
    global_ctr_threshold = fit.loc[fit["impressions_90d"] >= 100, "ctr"].median()

    scored["is_stale"] = scored["days_since_last_update"] >= 91
    scored["has_visibility"] = scored["impressions_90d"] >= visibility_threshold
    scored["has_high_demand"] = scored["search_volume"].fillna(0) >= demand_threshold
    scored["ctr_threshold_for_bucket"] = scored["position_tier"].map(position_ctr_threshold).fillna(global_ctr_threshold)
    scored["has_low_ctr"] = (
        scored["has_visibility"]
        & scored["avg_position"].gt(0)
        & scored["ctr"].le(scored["ctr_threshold_for_bucket"])
    )

    log_impressions = np.log1p(scored["impressions_90d"].fillna(0))
    max_log_impressions = log_impressions.max()
    scored["visibility_component"] = np.where(max_log_impressions > 0, log_impressions / max_log_impressions, 0)

    scored["baseline_score"] = (
        scored["is_stale"].astype(int) * 40
        + scored["has_low_ctr"].astype(int) * 30
        + scored["has_high_demand"].astype(int) * 20
        + scored["has_visibility"].astype(int) * 5
        + scored["visibility_component"] * 5
    )

    scored["reason_code"] = np.select(
        [scored["is_stale"], scored["has_low_ctr"], scored["has_high_demand"]],
        ["HIGH_STALENESS", "LOW_CTR", "HIGH_DEMAND"],
        default="LOWER_OPPORTUNITY",
    )

    threshold_frame = scored.loc[fit_rows.index]
    review_threshold = threshold_frame["baseline_score"].quantile(0.75)
    scored["baseline_prediction"] = (scored["baseline_score"] >= review_threshold).astype(int)

    return scored, {
        "demand_threshold": demand_threshold,
        "visibility_threshold": visibility_threshold,
        "review_threshold": review_threshold,
    }

baseline_scored, baseline_thresholds = add_week4_baseline_scores(df, train_df)
baseline_test = baseline_scored.loc[test_df.index].copy()

try:
    one_hot = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
except TypeError:
    one_hot = OneHotEncoder(handle_unknown="ignore", sparse=False)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", SimpleImputer(strategy="median"), [col for col in numeric_features if col in feature_cols]),
        ("cat", Pipeline(steps=[("imputer", SimpleImputer(strategy="most_frequent")), ("onehot", one_hot)]), [col for col in categorical_features if col in feature_cols]),
    ],
    remainder="drop",
)

model = Pipeline(
    steps=[
        ("preprocess", preprocessor),
        (
            "rf",
            RandomForestClassifier(
                n_estimators=200,
                max_depth=8,
                min_samples_leaf=20,
                random_state=RANDOM_STATE,
                n_jobs=-1,
                class_weight="balanced_subsample",
            ),
        ),
    ]
)

model.fit(X_train, y_train)

baseline_prediction = baseline_test["baseline_prediction"].to_numpy()
baseline_score = baseline_test["baseline_score"].to_numpy()
model_prediction = model.predict(X_test)
model_score = model.predict_proba(X_test)[:, 1]

comparison = pd.DataFrame(
    [
        {
            "Method": "Week-4 baseline",
            "Precision": precision_score(y_test, baseline_prediction, zero_division=0),
            "Recall": recall_score(y_test, baseline_prediction, zero_division=0),
            "F1": f1_score(y_test, baseline_prediction, zero_division=0),
            "ROC-AUC": roc_auc_score(y_test, baseline_score),
        },
        {
            "Method": "Week-5 Random Forest",
            "Precision": precision_score(y_test, model_prediction, zero_division=0),
            "Recall": recall_score(y_test, model_prediction, zero_division=0),
            "F1": f1_score(y_test, model_prediction, zero_division=0),
            "ROC-AUC": roc_auc_score(y_test, model_score),
        },
    ]
)

print("Week 4 baseline thresholds fitted on training rows:")
print({key: round(value, 4) for key, value in baseline_thresholds.items()})
print("\nModel-vs-baseline comparison on identical held-out rows:")
display(comparison)

model_f1 = comparison.loc[comparison["Method"].eq("Week-5 Random Forest"), "F1"].iloc[0]
model_auc = comparison.loc[comparison["Method"].eq("Week-5 Random Forest"), "ROC-AUC"].iloc[0]
if model_f1 >= 0.98 or model_auc >= 0.99:
    print("Warning: unusually high score. Re-check leakage audit and feature list before trusting this result.")
else:
    print("Score check: model performance is not suspiciously perfect.")


Week 4 baseline thresholds fitted on training rows:
{'demand_threshold': 20.0, 'visibility_threshold': 300, 'review_threshold': 57.3788}

Model-vs-baseline comparison on identical held-out rows:


,Method,Precision,Recall,F1,ROC-AUC
0,Week-4 baseline,0.522661,0.194558,0.283561,0.518877
1,Week-5 Random Forest,0.585040,0.668299,0.623904,0.605252


Score check: model performance is not suspiciously perfect.


## 5. Feature Interpretation

Permutation importance measures how much the model's F1 score changes when one input column is shuffled on the held-out test rows. Larger drops mean the model relied more heavily on that feature for distinguishing declining from non-declining content in this evaluation.

These importances are associations inside the trained model, not causal claims. They do not prove that changing a feature causes refresh success or decline.


In [5]:
perm = permutation_importance(
    model,
    X_test,
    y_test,
    scoring="f1",
    n_repeats=3,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

importance = (
    pd.DataFrame(
        {
            "feature": feature_cols,
            "mean_f1_drop_when_shuffled": perm.importances_mean,
            "std": perm.importances_std,
        }
    )
    .sort_values("mean_f1_drop_when_shuffled", ascending=False)
    .reset_index(drop=True)
)

display(importance.head(15))

top_features = importance.head(3)["feature"].tolist()
print("Top features by permutation importance:", top_features)
print(
    "Interpretation: these fields were most useful for distinguishing declining from non-declining content "
    "in the held-out clients. This is model reliance, not causation."
)


,feature,mean_f1_drop_when_shuffled,std
0,days_with_impressions,0.030290,0.004139
1,impressions_90d,0.007325,0.001447
2,avg_position,0.005225,0.002166
3,position_tier,0.003836,0.002521
4,ctr,0.002705,0.001637
5,impression_tier,0.002027,0.000061
6,char_count,0.001964,0.000992
7,sessions_90d,0.001542,0.000790
8,freshness_tier,0.001341,0.000313
9,users_90d,0.000695,0.000258


Top features by permutation importance: ['days_with_impressions', 'impressions_90d', 'avg_position']
Interpretation: these fields were most useful for distinguishing declining from non-declining content in the held-out clients. This is model reliance, not causation.


## 6. Error Analysis

The error review looks at false positives and false negatives directly. A false positive is a held-out item the model flags as declining when the proxy says it is not. A false negative is an item the proxy says is declining but the model misses.

These examples help show what signals may confuse the model and what context is missing, such as page intent, seasonality, SERP changes, recent edits that are not represented in the features, or whether a refresh would be worth the business effort.


In [6]:
error_analysis = test_df[["content_id", "content_type", "freshness_tier", "position_tier"]].copy()
for col in ["impressions_90d", "clicks_90d", "search_volume", "ctr", "avg_position", "days_since_last_update"]:
    error_analysis[col] = test_df[col]

error_analysis["actual"] = y_test.to_numpy()
error_analysis["predicted"] = model_prediction
error_analysis["model_probability"] = model_score
error_analysis["baseline_score"] = baseline_score
error_analysis["baseline_prediction"] = baseline_prediction
error_analysis["error_type"] = np.select(
    [
        (error_analysis["actual"] == 0) & (error_analysis["predicted"] == 1),
        (error_analysis["actual"] == 1) & (error_analysis["predicted"] == 0),
        (error_analysis["actual"] == 1) & (error_analysis["predicted"] == 1),
        (error_analysis["actual"] == 0) & (error_analysis["predicted"] == 0),
    ],
    ["FALSE_POSITIVE", "FALSE_NEGATIVE", "TRUE_POSITIVE", "TRUE_NEGATIVE"],
    default="UNKNOWN",
)

print("Confusion matrix [[TN, FP], [FN, TP]]:")
print(confusion_matrix(y_test, model_prediction))
print("\nError counts:")
print(error_analysis["error_type"].value_counts())

print("\nFalse positives - model recommended decline/refresh review, proxy did not:")
display(
    error_analysis[error_analysis["error_type"].eq("FALSE_POSITIVE")]
    .sort_values("model_probability", ascending=False)
    .head(8)
)

print("\nFalse negatives - proxy says declining, model missed:")
display(
    error_analysis[error_analysis["error_type"].eq("FALSE_NEGATIVE")]
    .sort_values("model_probability", ascending=True)
    .head(8)
)

error_summary = (
    error_analysis
    .groupby("error_type", observed=False)
    .agg(
        rows=("content_id", "size"),
        median_impressions_90d=("impressions_90d", "median"),
        median_ctr=("ctr", "median"),
        median_avg_position=("avg_position", "median"),
        median_days_since_last_update=("days_since_last_update", "median"),
        median_model_probability=("model_probability", "median"),
    )
    .reset_index()
)

print("\nError pattern summary:")
display(error_summary)

fp_count = int((error_analysis["error_type"] == "FALSE_POSITIVE").sum())
fn_count = int((error_analysis["error_type"] == "FALSE_NEGATIVE").sum())
main_error_pattern = (
    "False positives are more common than false negatives."
    if fp_count > fn_count
    else "False negatives are more common than false positives."
    if fn_count > fp_count
    else "False positives and false negatives are balanced."
)
print("\nMain error pattern:", main_error_pattern)
print(
    "Likely missing context: query intent, seasonality, SERP changes, recent editorial changes, and whether the page is worth refreshing. "
    "A recommendation can be wrong when the decline proxy reflects noise or when a stale/high-opportunity page is intentionally evergreen."
)


Confusion matrix [[TN, FP], [FN, TP]]:
[[1698 1742]
 [1219 2456]]

Error counts:
error_type
TRUE_POSITIVE     2456
FALSE_POSITIVE    1742
TRUE_NEGATIVE     1698
FALSE_NEGATIVE    1219
Name: count, dtype: int64

False positives - model recommended decline/refresh review, proxy did not:


,content_id,content_type,freshness_tier,position_tier,impressions_90d,clicks_90d,search_volume,ctr,avg_position,days_since_last_update,actual,predicted,model_probability,baseline_score,baseline_prediction,error_type
11061,content_0b47dae0c7f9,keyword article,91-180,page_3_5,1191,0,20.0,0.00,23.1,103,0,1,0.826826,97.691833,1,FALSE_POSITIVE
5477,content_3164f3076003,keyword article,91-180,striking,2696,1,110.0,0.04,16.1,104,0,1,0.820713,98.002123,1,FALSE_POSITIVE
22526,content_1d0963b56227,keyword article,91-180,page_3_5,3445,3,20.0,0.09,39.0,104,0,1,0.816301,68.095256,1,FALSE_POSITIVE
5011,content_c148e44db30d,keyword article,91-180,page_3_5,335,0,20.0,0.00,31.3,104,0,1,0.815984,97.210622,1,FALSE_POSITIVE
10080,content_35d63627bf3e,keyword article,91-180,page_3_5,1525,0,20.0,0.00,32.6,103,0,1,0.815966,97.785705,1,FALSE_POSITIVE
13,content_a5a2fbc76336,keyword article,91-180,page_3_5,307,0,10.0,0.00,39.8,103,0,1,0.814279,77.177556,1,FALSE_POSITIVE
22042,content_2ba626fea4d6,keyword article,91-180,page_1,360,0,10.0,0.00,7.2,104,0,1,0.813104,77.237895,1,FALSE_POSITIVE
22524,content_846bb4dd8b44,keyword article,91-180,striking,870,1,10.0,0.11,17.6,104,0,1,0.809148,77.572603,1,FALSE_POSITIVE



False negatives - proxy says declining, model missed:


,content_id,content_type,freshness_tier,position_tier,impressions_90d,clicks_90d,search_volume,ctr,avg_position,days_since_last_update,actual,predicted,model_probability,baseline_score,baseline_prediction,error_type
1864,content_16f38acf0f26,keyword article,0-30,page_3_5,2,0,20.0,0.0,50.0,20,1,0,0.113512,20.417495,0,FALSE_NEGATIVE
1725,content_8c482a64a3df,keyword article,0-30,top_3,1,0,0.0,0.0,3.0,20,1,0,0.146349,0.263410,0,FALSE_NEGATIVE
2976,content_9de9afdada19,keyword article,0-30,page_1,1,0,720.0,0.0,8.0,20,1,0,0.149012,20.263410,0,FALSE_NEGATIVE
18423,content_77e2a54525b6,keyword article,0-30,page_1,1,0,10.0,0.0,7.0,20,1,0,0.150196,0.263410,0,FALSE_NEGATIVE
27271,content_7bc32bc1df59,keyword article,91-180,top_3,1,0,20.0,0.0,0.0,92,1,0,0.151291,60.263410,1,FALSE_NEGATIVE
22285,content_f85aa6e9bc6e,keyword article,0-30,page_3_5,2,0,10.0,0.0,21.5,20,1,0,0.159709,0.417495,0,FALSE_NEGATIVE
27395,content_e72e6c56f0a3,keyword article,0-30,page_1,1,0,10.0,0.0,9.0,20,1,0,0.164272,0.263410,0,FALSE_NEGATIVE
23915,content_823ed1f9a2fe,keyword article,0-30,page_1,2,0,20.0,0.0,7.0,20,1,0,0.166451,20.417495,0,FALSE_NEGATIVE



Error pattern summary:


,error_type,rows,median_impressions_90d,median_ctr,median_avg_position,median_days_since_last_update,median_model_probability
0,FALSE_NEGATIVE,1219,831.0,0.05,12.90,20.0,0.403807
1,FALSE_POSITIVE,1742,177.5,0.00,9.65,20.0,0.625289
2,TRUE_NEGATIVE,1698,525.0,0.00,10.50,20.0,0.381820
3,TRUE_POSITIVE,2456,214.5,0.00,8.20,20.0,0.646588



Main error pattern: False positives are more common than false negatives.
Likely missing context: query intent, seasonality, SERP changes, recent editorial changes, and whether the page is worth refreshing. A recommendation can be wrong when the decline proxy reflects noise or when a stale/high-opportunity page is intentionally evergreen.


## 7. Self-Check

- [x] Method choice and reasoning are explained.
- [x] Target/proxy is clearly defined.
- [x] No target leakage.
- [x] Features are available at decision time.
- [x] Valid grouped validation design is used.
- [x] Baseline and model use the same evaluation population.
- [x] Week-4 baseline is used, not a majority-class baseline.
- [x] Same metrics are used for baseline and model.
- [x] Model metrics are reported.
- [x] Model-vs-baseline table is present.
- [x] Features are interpreted.
- [x] Errors are examined.
- [x] Complexity is not rewarded merely for being complex.
- [x] Notebook runs from top to bottom.


In [7]:
baseline_f1 = comparison.loc[comparison["Method"].eq("Week-4 baseline"), "F1"].iloc[0]
model_f1 = comparison.loc[comparison["Method"].eq("Week-5 Random Forest"), "F1"].iloc[0]
baseline_auc = comparison.loc[comparison["Method"].eq("Week-4 baseline"), "ROC-AUC"].iloc[0]
model_auc = comparison.loc[comparison["Method"].eq("Week-5 Random Forest"), "ROC-AUC"].iloc[0]

print("FINAL REPORT")
print("1. Final target definition: is_declining_label = 1 when trend_direction == 'down', else 0.")
print(f"2. Final feature list ({len(feature_cols)} features): {feature_cols}")
print("3. Split/validation design: GroupShuffleSplit by client_id; held-out clients are used only for testing.")
print(f"4. Week-4 baseline score: F1={baseline_f1:.3f}, ROC-AUC={baseline_auc:.3f}.")
print(f"5. Week-5 model score: F1={model_f1:.3f}, ROC-AUC={model_auc:.3f}.")
print(f"6. Whether the model beats the baseline on F1: {model_f1 > baseline_f1}.")
print(f"7. Main error pattern: {main_error_pattern}")
print("8. Leakage checks passed: target, target-source, future-window, label-derived, and ID columns were excluded from X.")


FINAL REPORT
1. Final target definition: is_declining_label = 1 when trend_direction == 'down', else 0.
2. Final feature list (32 features): ['search_volume', 'competition', 'cpc', 'word_count', 'char_count', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'content_age_days', 'age_tier_order', 'days_since_last_update', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'competition_level', 'content_type', 'main_intent', 'age_tier', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'impression_tier', 'position_tier']
3. Split/validation design: GroupShuffleSplit by client_id; held-out clients are used only for testing.
4. Week-4 baseline score: F1=0.284, ROC-AUC=0.519.
5. Week-5 model score: F1=0.624, ROC-AUC=0.605.
6. Whether the model beats the baseline on F1: True.
7. Main error pattern: False positives are more co